<a href="https://colab.research.google.com/github/norayyh/AutoInsight-R/blob/main/autoinsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install anthropic pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 15.3 MB/s eta 0:00:00


In [6]:
# Upload datasets to Colab from GitHub
!wget -q https://raw.githubusercontent.com/norayyh/AutoInsight-R/main/datasets/Iris.csv -O Iris.csv
!wget -q https://raw.githubusercontent.com/norayyh/AutoInsight-R/main/datasets/titanic.csv -O titanic.csv
!wget -q https://raw.githubusercontent.com/norayyh/AutoInsight-R/main/datasets/housing.csv -O housing.csv
print("Datasets downloaded successfully")

Datasets downloaded successfully


In [2]:
# Install the Anthropic library
import anthropic
from google.colab import userdata

# Initialize the client using the secret API key
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

# Send a test request to verify the API connection
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Write Python code using pandas to read iris.csv and print the first 5 rows"}
    ]
)

print(response.content[0].text)

## Reading Iris Dataset with Pandas

```python
import pandas as pd

# Read the CSV file
df = pd.read_csv('iris.csv')

# Print the first 5 rows
print(df.head())
```

### Example Output:
```
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa
```

---

### Alternative: Load directly from seaborn (no CSV file needed)
```python
import pandas as pd
import seaborn as sns

# Load iris dataset directly from seaborn
df = sns.load_dataset('iris')

# Print the first 5 rows
print(df.head())
```

---

### Key Methods Used:
| Method | Description |
|--------|-------------|
| `pd.read_csv('iris.csv')` | Reads a CSV file into a DataFrame |
| `df.head()` | Returns the **first 

In [3]:
import traceback

def execute_code(code: str) -> tuple[bool, str]:
    """
    Execute the generated Python code and capture the result or error.
    Returns (success, message).
    """
    try:
        exec(code, {})
        return True, "Code executed successfully"
    except Exception as e:
        return False, traceback.format_exc()

# Test the executor with a simple example
test_code = "print('Executor is working!')"
success, message = execute_code(test_code)
print(message)

Executor is working!
Code executed successfully


In [8]:
import re

def clean_code(code: str) -> str:
    """
    Remove markdown code block formatting from generated code.
    """
    code = re.sub(r"```python\n", "", code)
    code = re.sub(r"```", "", code)
    return code.strip()


def generate_code(user_request: str, dataset_path: str) -> str:
    """
    Send a request to Claude to generate Python code for data analysis.
    """
    prompt = f"""You are a data analysis assistant.
The dataset is located at: {dataset_path}
Write Python code to complete the following task: {user_request}
Only return the Python code, no explanations or markdown formatting.
Only use these libraries: pandas, matplotlib, seaborn, sklearn.
"""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


def repair_code(user_request: str, previous_code: str, error_message: str, dataset_path: str) -> str:
    """
    Send the failed code and error message back to Claude for repair.
    """
    prompt = f"""You are a data analysis assistant.
The dataset is located at: {dataset_path}
The user request was: {user_request}

The following code was generated but failed:
{previous_code}

The error message was:
{error_message}

Please fix the code. Only return the corrected Python code, no explanations or markdown formatting.
Only use these libraries: pandas, matplotlib, seaborn, sklearn.
"""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


def execute_code(code: str) -> tuple[bool, str]:
    """
    Execute the generated Python code and capture the result or error.
    Returns (success, message).
    """
    try:
        exec(code, {})
        return True, "Code executed successfully"
    except Exception as e:
        return False, traceback.format_exc()


def run_pipeline(user_request: str, dataset_path: str, max_retries: int = 3) -> dict:
    """
    Run the full self-healing pipeline:
    generate code -> execute -> repair if failed -> retry.
    """
    code = clean_code(generate_code(user_request, dataset_path))

    for attempt in range(max_retries):
        success, message = execute_code(code)

        if success:
            print(f"Success on attempt {attempt + 1}")
            return {"success": True, "attempts": attempt + 1, "code": code}

        print(f"Attempt {attempt + 1} failed. Repairing...")

        if attempt < max_retries - 1:
            code = clean_code(repair_code(user_request, code, message, dataset_path))

    return {"success": False, "attempts": max_retries, "code": code}


# Test the pipeline with an easy task
result = run_pipeline(
    user_request="Calculate the mean, standard deviation, max, and min for each numeric feature",
    dataset_path="Iris.csv"
)
print(result)

              Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
mean   75.500000       5.843333      3.054000       3.758667      1.198667
std    43.445368       0.828066      0.433594       1.764420      0.763161
max   150.000000       7.900000      4.400000       6.900000      2.500000
min     1.000000       4.300000      2.000000       1.000000      0.100000
Success on attempt 1
{'success': True, 'attempts': 1, 'code': "import pandas as pd\n\ndf = pd.read_csv('Iris.csv')\n\nnumeric_df = df.select_dtypes(include='number')\n\nstats = numeric_df.agg(['mean', 'std', 'max', 'min'])\n\nprint(stats)"}
